In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import graph_tool.all as gt

# ==========================================
# 1. Extract directed transition probabilities
# ==========================================
file_path = '/Users/jiaxin/Downloads/masked_network_Rcmems_Pico11_S2017-8-1_D90_DT20_ODT24_Cico7.net'

edges = {}
nodes = set()

with open(file_path, 'r') as f:
    parsing_arcs = False
    for line in f:
        line = line.strip()
        if line.lower().startswith('*arcs'):
            parsing_arcs = True
            continue
        elif line.lower().startswith('*vertices') or not parsing_arcs or not line:
            continue
        
        parts = line.split()
        if len(parts) >= 3:
            u = int(parts[0])
            v = int(parts[1])
            w = float(parts[2])
            
            # Filter self-loops
            if u == v:
                continue
                
            edges[(u, v)] = w
            nodes.add(u)
            nodes.add(v)

print(f"-> {len(nodes)} nodes, {len(edges)} edges.")

# ==========================================
# 2. Calculate flow
# ==========================================
print("\nF(x,y) = W(x,y) - W(y,x)...")
net_flows = []
processed_pairs = set()

for (u, v), w_uv in edges.items():
    node1, node2 = min(u, v), max(u, v)
    if (node1, node2) in processed_pairs:
        continue
    w_12 = edges.get((node1, node2), 0.0)
    w_21 = edges.get((node2, node1), 0.0)
    
    # F(x,y) = W(x,y) - W(y,x)
    net_flow = w_12 - w_21
    
    if net_flow != 0:
        net_flows.append({
            'node1': node1,
            'node2': node2,
            'flow': net_flow
        })
    processed_pairs.add((node1, node2))

df_net = pd.DataFrame(net_flows)
df_net['abs_flow'] = df_net['flow'].abs()

csv_filename_flow = 'arctic_ocean_flow.csv'
df_net[['node1', 'node2', 'flow']].to_csv(csv_filename_flow, index=False)
print(f"-> Flow file saved to: {csv_filename_flow}")

# ==========================================
# 3. EDA
# ==========================================
sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Figure A: Original distribution of flow
sns.histplot(df_net['flow'], bins=100, ax=axes[0], color='teal', kde=False)
axes[0].set_title('Distribution of Flow')
axes[0].set_xlabel('Flow (F > 0 implies node1 -> node2)')
axes[0].set_ylabel('Frequency (Log Scale)')
axes[0].set_yscale('log')

# Figure B: Distribution of Absolute Flow
sns.histplot(df_net['abs_flow'], bins=100, ax=axes[1], color='coral', kde=False)
axes[1].set_title('Distribution of Absolute Flow Magnitude (|F|)')
axes[1].set_xlabel('Absolute Magnitude of Flow')
axes[1].set_ylabel('Frequency (Log Scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

# ==========================================
# 4. Build network and calculate Edge Degree & Betweenness
# ==========================================
print("\n4. Building undirected graph and calculating Edge Metrics...")

# Double insurance: ensure node1 is strictly less than node2
swap_mask = df_net['node1'] > df_net['node2']
df_net.loc[swap_mask, ['node1', 'node2']] = df_net.loc[swap_mask, ['node2', 'node1']].values
df_net.loc[swap_mask, 'flow'] = -df_net.loc[swap_mask, 'flow']

# Build unweighted, undirected graph
g = gt.Graph(directed=False)
# Use .tolist() to prevent numpy compatibility issues, and string for alphanumeric IDs
g.vp.ids = g.add_edge_list(df_net[['node1', 'node2']].values, 
                           hashed=True, 
                           hash_type='string')

# --- Calculate Edge Degree ---
deg = g.degree_property_map("total")
node_degrees_dict = {g.vp.ids[v]: deg[v] for v in g.vertices()}
df_net['edge_degree'] = df_net['node1'].map(node_degrees_dict) + df_net['node2'].map(node_degrees_dict)
print("-> Edge degree calculation completed.")

# --- Calculate Edge Betweenness ---
print("-> Calculating topological Edge Betweenness...")
_, e_bet = gt.betweenness(g, norm=True)
# .fa extracts the flat array perfectly matching the edge insertion order
df_net['edge_betweenness'] = e_bet.fa 
print("-> Edge betweenness calculation completed.")

# --- Visualize Distributions (Degree & Betweenness) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Degree Histogram
axes[0].hist(df_net['edge_degree'], bins=50, log=True, color='purple', edgecolor='black')
axes[0].set_title('Edge Degree Distribution', fontsize=14)
axes[0].set_xlabel('Edge Degree (Sum of Node Degrees)', fontsize=12)
axes[0].set_ylabel('Frequency (Log Scale)', fontsize=12)

# Betweenness Histogram
axes[1].hist(df_net['edge_betweenness'], bins=50, log=True, color='green', edgecolor='black')
axes[1].set_title('Edge Betweenness Distribution', fontsize=14)
axes[1].set_xlabel('Edge Betweenness Centrality', fontsize=12)
axes[1].set_ylabel('Frequency (Log Scale)', fontsize=12)

plt.tight_layout()
plt.show()

df_final = df_net.sort_values(by='edge_degree', ascending=False)
df_final = df_final[['node1', 'node2', 'flow', 'edge_degree', 'edge_betweenness']]

csv_filename_deg = 'arctic_ocean_edge_degree.csv'
df_final.to_csv(csv_filename_deg, index=False)

In [ ]:
# [Cell 1: Imports & Setup]
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import graph_tool.all as gt

# 动态定位项目根目录 (向上寻找包含 'data' 文件夹的目录)
current_dir = os.path.abspath('')
project_root = current_dir
while not os.path.isdir(os.path.join(project_root, 'data')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root:
        raise FileNotFoundError("找不到项目根目录！请确保你在 Edge-Flow-Hypothesis-Tests 项目内运行。")
    project_root = parent_dir

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Project root dynamically resolved to: {project_root}")

In [ ]:
# [Cell 2: Define Paths & Load Raw Data]
# 定义输入输出目录
raw_data_dir = os.path.join(project_root, 'data', 'real_wolrd', 'raw')
preprocessed_dir = os.path.join(project_root, 'data', 'real_wolrd', 'preprocessed')

# 确保预处理输出文件夹存在
os.makedirs(preprocessed_dir, exist_ok=True)

# 你的原始数据文件名
input_filename = 'masked_network_Rcmems_Pico11_S2017-8-1_D90_DT20_ODT24_Cico7.net'
file_path = os.path.join(raw_data_dir, input_filename)

print(f"📂 Reading raw data from: {file_path}")

# ==========================================
# 提取有向转移概率
# ==========================================
edges = {}
nodes = set()

with open(file_path, 'r') as f:
    parsing_arcs = False
    for line in f:
        line = line.strip()
        if line.lower().startswith('*arcs'):
            parsing_arcs = True
            continue
        elif line.lower().startswith('*vertices') or not parsing_arcs or not line:
            continue
        
        parts = line.split()
        if len(parts) >= 3:
            u = int(parts[0])
            v = int(parts[1])
            w = float(parts[2])
            
            # Filter self-loops
            if u == v:
                continue
                
            edges[(u, v)] = w
            nodes.add(u)
            nodes.add(v)

print(f"-> 成功提取: {len(nodes)} nodes, {len(edges)} edges.")

In [ ]:
# [Cell 3: Calculate Net Flow]
print("Calculating F(x,y) = W(x,y) - W(y,x)...")
net_flows = []
processed_pairs = set()

for (u, v), w_uv in edges.items():
    source, target = min(u, v), max(u, v)
    if (source, target) in processed_pairs:
        continue
        
    w_12 = edges.get((source, target), 0.0)
    w_21 = edges.get((target, source), 0.0)
    
    # F(x,y) = W(x,y) - W(y,x)
    net_flow = w_12 - w_21
    
    if net_flow != 0:
        net_flows.append({
            'source': source,
            'target': target,
            'flow': net_flow
        })
    processed_pairs.add((source, target))

df_net = pd.DataFrame(net_flows)
df_net['abs_flow'] = df_net['flow'].abs()

print(f"-> 流计算完成，共生成 {len(df_net)} 条净流边。")

In [ ]:
# [Cell 4: Exploratory Data Analysis]
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Figure A: Original distribution of flow
sns.histplot(df_net['flow'], bins=100, ax=axes[0], color='teal', kde=False)
axes[0].set_title('Distribution of Net Flow')
axes[0].set_xlabel('Flow (F > 0 implies source -> target)')
axes[0].set_ylabel('Frequency (Log Scale)')
axes[0].set_yscale('log')

# Figure B: Distribution of Absolute Flow
sns.histplot(df_net['abs_flow'], bins=100, ax=axes[1], color='coral', kde=False)
axes[1].set_title('Distribution of Absolute Flow Magnitude (|F|)')
axes[1].set_xlabel('Absolute Magnitude of Flow')
axes[1].set_ylabel('Frequency (Log Scale)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

In [ ]:
# [Cell 5: Network Topologies & Export]
print("Building undirected graph and calculating Edge Metrics...")

# Double insurance: ensure source is strictly less than target
swap_mask = df_net['source'] > df_net['target']
df_net.loc[swap_mask, ['source', 'target']] = df_net.loc[swap_mask, ['target', 'source']].values
df_net.loc[swap_mask, 'flow'] = -df_net.loc[swap_mask, 'flow']

# Build unweighted, undirected graph using graph-tool
g = gt.Graph(directed=False)
g.vp.ids = g.add_edge_list(df_net[['source', 'target']].values, 
                           hashed=True, 
                           hash_type='string')

# --- Calculate Edge Degree ---
deg = g.degree_property_map("total")
node_degrees_dict = {g.vp.ids[v]: deg[v] for v in g.vertices()}
df_net['edge_degree'] = df_net['source'].map(node_degrees_dict) + df_net['target'].map(node_degrees_dict)

# --- Calculate Edge Betweenness ---
print("Calculating topological Edge Betweenness...")
_, e_bet = gt.betweenness(g, norm=True)
df_net['edge_betweenness'] = e_bet.fa 

# --- Visualize Distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_net['edge_degree'], bins=50, log=True, color='purple', edgecolor='black')
axes[0].set_title('Edge Degree Distribution')
axes[1].hist(df_net['edge_betweenness'], bins=50, log=True, color='green', edgecolor='black')
axes[1].set_title('Edge Betweenness Distribution')
plt.tight_layout()
plt.show()

# ==========================================
# 导出数据到 preprocessed 文件夹
# ==========================================
df_final = df_net.sort_values(by='edge_degree', ascending=False)
df_final = df_final[['source', 'target', 'flow', 'edge_degree', 'edge_betweenness']]

# 我们统一将其命名为下游调用的文件： ocean_flow_data.csv
output_csv_filename = 'ocean_flow_data.csv'
final_output_path = os.path.join(preprocessed_dir, output_csv_filename)

df_final.to_csv(final_output_path, index=False)
print(f"\n✅ All done! Preprocessed data strictly saved to: {final_output_path}")